# Projeto: Detecção de Fraude em Planos de Saúde

## 1. Contexto do Problema
A fraude na saúde representa de **3% a 10%** dos gastos totais do setor. Este dataset simula 10.000 reivindicações (claims) reais, onde o comportamento fraudulento muitas vezes se camufla em atividades médicas legítimas.

---

## 2. Dicionário de Dados

### Informações Financeiras e do Claim
* `Claim_Amount`: Valor solicitado.
* `Approved_Amount`: Valor aprovado pelo plano.
* `Claim_Status`: Status da solicitação (Aprovada, Rejeitada, Pendente).
* `Insurance_Type`: Tipo de seguro (Private, Medicare, Medicaid, Self-pay).

### Informações do Paciente
* `Patient_Age` / `Gender`: Idade e gênero.
* `State`: Localização geográfica.
* `Chronic_Condition`: Indicador de condição crônica.
* `Prior_Visit_History`: Histórico de visitas nos últimos 12 meses.

### Informações do Provedor (Médico/Hospital)
* `Provider_ID`: Identificador único do provedor.
* `Provider_Specialty`: Especialidade (Cardiologia, Ortopedia, Clínica Geral, etc.).
* `Monthly_Claim_Volume`: Volume mensal de solicitações por provedor.

### Codificação Médica
* `Diagnosis_Codes`: Códigos ICD-10 de diagnóstico.
* `Procedure_Codes`: Códigos CPT de procedimentos.

### Atributos Temporais e Adicionais
* `Claim_Submission_Date`: Data de submissão.
* `Days_to_Submission`: Dias entre o serviço e a submissão da cobrança.
* `Length_of_Stay`: Duração da internação (para casos Inpatient).
* `Visit_Type`: Tipo de visita (Outpatient, Inpatient, Emergency).

---

## 3. Variável Alvo (Target)
* **`Is_Fraud`**: 
    * `0`: Reivindicação Legítima.
    * `1`: Reivindicação Fraudulenta.
* **Taxa de Fraude**: Aproximadamente **8%** (Classe desbalanceada).

---

## 4. Padrões de Fraude Esperados
1. Reivindicações fraudulentas tendem a ter **valores mais altos**.
2. Casos de fraude costumam ser submetidos com **menor atraso** após o serviço.
3. Certos provedores concentram maior atividade suspeita.

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#removendo os limites de visualização
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.options.display.float_format = '{:.2f}'.format


In [21]:
#importando o meu dataset
path = r"C:\Users\User\Desktop\projetos\projeto-fraude-saude\data\raw-data\healthcare_fraud_detection.csv"
df = pd.read_csv(path)

## 1 Avaliando o conjunto de dados de forma macro para compreender os dados

In [22]:
df.head()

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.00
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.00
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.00
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.00
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.00


In [23]:
#Quantas linhas e colunas tem no dataframe
df.shape

(10000, 20)

In [24]:
#Tipos de dado de cada coluna
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Provider_ID                            10000 non-null  object 
 1   Claim_ID                               10000 non-null  object 
 2   Patient_Age                            10000 non-null  int64  
 3   Patient_Gender                         10000 non-null  object 
 4   Diagnosis_Code                         10000 non-null  object 
 5   Procedure_Code                         10000 non-null  int64  
 6   Claim_Amount                           10000 non-null  float64
 7   Approved_Amount                        10000 non-null  float64
 8   Insurance_Type                         9650 non-null   object 
 9   Claim_Submission_Date                  10000 non-null  object 
 10  Days_Between_Service_and_Claim         10000 non-null  int64  
 11  Num

## 2 Verificar se existem valores nulos nos dados


In [25]:
df.isnull().sum()

Provider_ID                                0
Claim_ID                                   0
Patient_Age                                0
Patient_Gender                             0
Diagnosis_Code                             0
Procedure_Code                             0
Claim_Amount                               0
Approved_Amount                            0
Insurance_Type                           350
Claim_Submission_Date                      0
Days_Between_Service_and_Claim             0
Number_of_Claims_Per_Provider_Monthly      0
Provider_Specialty                       350
Patient_State                              0
Claim_Status                               0
Is_Fraud                                   0
Length_of_Stay                             0
Visit_Type                                 0
Chronic_Condition_Flag                     0
Prior_Visits_12m                         350
dtype: int64

Insight de Engenharia de Dados: O Padrão dos Nulos Coincidentes

Aqui temos a primeira informação importante do projeto. As colunas Insurance_Type, Provider_Specialty, Prior_Visits_12m tem exatamente 350 valores nulos e isso não parece ser coincidência. Geralmente quando o número nulo "bate" perfeitamente entre colunas, isso indica um padrão sistêmico.

As colunas que estão nulas são:

* `Insurance_Type` (Tipo de Seguro)
* `Provider_Specialty` (Especialidade do Médico)
* `Prior_Visits_12m` (Histórico de Visitas)

Fraudadores muitas vezes tentam criar "contas fantasma" ou registrar procedimentos para pacientes que não existem ou que não frequentam aquela clínica. A falta de especialidade do provedor ou do histórico de visitas do paciente pode indicar que esses registros foram gerados artificialmente no sistema de cobrança, sem uma base de dados real por trás para preencher esses campos automáticos.

In [29]:
# Tratando as colunas de texto (Categorias)
df['Provider_Specialty'] = df['Provider_Specialty'].fillna('UNKNOWN')
df['Insurance_Type'] = df['Insurance_Type'].fillna('UNKNOWN')

# Tratando a coluna numérica
# Aqui, em vez de UNKNOWN (que é texto), usamos -1 para manter a coluna como número
# mas indicar que o valor original era inexistente.
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(-1)

# Verificando se ainda resta algum nulo
print(df.isnull().sum())

Provider_ID                              0
Claim_ID                                 0
Patient_Age                              0
Patient_Gender                           0
Diagnosis_Code                           0
Procedure_Code                           0
Claim_Amount                             0
Approved_Amount                          0
Insurance_Type                           0
Claim_Submission_Date                    0
Days_Between_Service_and_Claim           0
Number_of_Claims_Per_Provider_Monthly    0
Provider_Specialty                       0
Patient_State                            0
Claim_Status                             0
Is_Fraud                                 0
Length_of_Stay                           0
Visit_Type                               0
Chronic_Condition_Flag                   0
Prior_Visits_12m                         0
dtype: int64


## 3 Verificar os valores únicos em cada variável

In [ ]:
valores_unicos = []

for i in df.columns[0:19].to_list():
    print(i,":")